In [1]:
# Fix NumPy compatibility (run in a separate cell, then restart kernel)
!pip install "numpy<2"


Defaulting to user installation because normal site-packages is not writeable


In [2]:
!pip install nest_asyncio


Defaulting to user installation because normal site-packages is not writeable


In [ ]:
# =============================================================================
# 🔧 STEP 1: INSTALL REQUIRED PACKAGES
# =============================================================================
# Run this cell first to install dependencies

!pip install playwright
!playwright install chromium


In [2]:
# =============================================================================
# 🔧 STEP 2: DOWNLOAD THE REPOSITORY (No Git Required) - CLEAN
# =============================================================================
import os
import urllib.request
import zipfile
import shutil

# Set your desktop path
desktop_path = os.path. expanduser("~/Desktop")
repo_dir = os.path. join(desktop_path, "scrape_chinese_social_media")

# Download and extract if it doesn't exist
if not os.path.exists(repo_dir):
    print("Downloading repository...")
    
    # GitHub ZIP download URL - completely clean
    zip_url = "https://github.com/pwklam/chinese-social-media-scrape-updated/archive/refs/heads/main.zip"
    zip_path = os.path.join(desktop_path, "repo_temp. zip")
    
    try:
        # Download the ZIP file
        urllib.request.urlretrieve(zip_url, zip_path)
        print("Download complete!")
        
        # Extract the ZIP file
        print("Extracting files...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref: 
            zip_ref.extractall(desktop_path)
        
        # Rename the extracted folder
        extracted_folder = os.path.join(desktop_path, "chinese-social-media-scrape-updated-main")
        if os.path.exists(extracted_folder):
            shutil.move(extracted_folder, repo_dir)
        
        # Clean up ZIP file
        os. remove(zip_path)
        
        print(f"Repository downloaded to {repo_dir}")
        
    except Exception as e: 
        print(f"Error downloading:  {e}")
else:
    print(f"Repository already exists at {repo_dir}")

# Verify the directory structure
if os.path.exists(repo_dir):
    print(f"\nContents of {repo_dir}:")
    for item in os.listdir(repo_dir):
        print(f"   - {item}")

Repository already exists at C:\Users\wk2lam/Desktop\scrape_chinese_social_media

Contents of C:\Users\wk2lam/Desktop\scrape_chinese_social_media:
   - duoyin
   - weibo


In [3]:
# =============================================================================
# 🔧 CELL 3: CONFIGURE URLS TO SCRAPE
# =============================================================================
# Add your Douyin URLs here
douyin_urls = """
https://www.iesdouyin.com/share/video/6862533859125251340/?region=CN&mid=6862534011185892109&u_code=0&did=MS4wLjABAAAA_oswJqZmU3zCGUyu8OVAC1UU2mCfmc4viCaQE_FNvdMR9u0Icg6C4KNfO5gmMhFF&iid=MS4wLjABAAAA0wvEGyff1ADjiPCblRQIWtvkG7eNV8LYyZ5_BPkYmSd4qCnN--z9LfPLty-TjygW&with_sec_did=1&titleType=title&share_sign=n02R2vZ5QCppS8kyhh0M5mtEAoEqelWcGWdKq65lS64-&share_version=110900&ts=1716365321&from_aid=2955&from_ssr=1
https://www.iesdouyin.com/share/video/7124980639866064136/?region=CN&mid=7124980721214688008&u_code=0&did=MS4wLjABAAAAFs6wBLkYsgEPdS0toRIdQ4FNZVrRttrZ-gldZ-56LSK3eaHcPqyi3HGlBt4b5WY5&iid=MS4wLjABAAAAcvmrmllesJPQC9ZqkE5KrO3Pltag8SfeuErDYO_x-KbAD5NpQEh0Owx11QG1GDhO&with_sec_did=1&titleType=title&share_sign=F7KrmxLyM3IblwEJJ_M6lglp5BE8bB9DnZKXa92CDkU-&share_version=110900&ts=1716365931&from_aid=2955&from_ssr=1"""

with open('urls.txt', 'w', encoding='utf-8') as f:
    f.write(douyin_urls. strip())

print("✅ URLs saved to urls.txt")
print("📝 Current URLs:")
with open('urls.txt', 'r', encoding='utf-8') as f:
    print(f.read())


✅ URLs saved to urls.txt
📝 Current URLs:
https://www.iesdouyin.com/share/video/6862533859125251340/?region=CN&mid=6862534011185892109&u_code=0&did=MS4wLjABAAAA_oswJqZmU3zCGUyu8OVAC1UU2mCfmc4viCaQE_FNvdMR9u0Icg6C4KNfO5gmMhFF&iid=MS4wLjABAAAA0wvEGyff1ADjiPCblRQIWtvkG7eNV8LYyZ5_BPkYmSd4qCnN--z9LfPLty-TjygW&with_sec_did=1&titleType=title&share_sign=n02R2vZ5QCppS8kyhh0M5mtEAoEqelWcGWdKq65lS64-&share_version=110900&ts=1716365321&from_aid=2955&from_ssr=1
https://www.iesdouyin.com/share/video/7124980639866064136/?region=CN&mid=7124980721214688008&u_code=0&did=MS4wLjABAAAAFs6wBLkYsgEPdS0toRIdQ4FNZVrRttrZ-gldZ-56LSK3eaHcPqyi3HGlBt4b5WY5&iid=MS4wLjABAAAAcvmrmllesJPQC9ZqkE5KrO3Pltag8SfeuErDYO_x-KbAD5NpQEh0Owx11QG1GDhO&with_sec_did=1&titleType=title&share_sign=F7KrmxLyM3IblwEJJ_M6lglp5BE8bB9DnZKXa92CDkU-&share_version=110900&ts=1716365931&from_aid=2955&from_ssr=1


In [4]:
# =============================================================================
# 🚀 CELL 4: RUN DOUYIN SCRAPER (COMPLETE FIXED VERSION)
# =============================================================================
import os
import sys
import asyncio
import sqlite3
from datetime import datetime
from playwright.async_api import async_playwright
import json

# Install nest_asyncio if not already installed
try:
    import nest_asyncio
except ImportError:
    import subprocess
    subprocess.check_call([sys. executable, "-m", "pip", "install", "nest_asyncio"])
    import nest_asyncio

nest_asyncio.apply()

# =============================================================================
# SET UP CORRECT WORKING DIRECTORY
# =============================================================================
desktop_path = os.path. expanduser("~/Desktop")
repo_dir = os.path. join(desktop_path, "scrape_chinese_social_media")

# Find the correct scraper directory (duoyin folder)
scraper_dir = os.path.join(repo_dir, "duoyin", "scrape_chinese_social_media")

if not os.path. exists(scraper_dir):
    # Try alternative paths
    possible_paths = [
        os.path. join(repo_dir, "duoyin", "scrape_chinese_social_media"),
        os.path.join(repo_dir, "weibo", "scrape-social-media-main"),
        repo_dir,
    ]
    for path in possible_paths:
        if os.path.exists(path) and os.path. exists(os.path. join(path, "utils. py")):
            scraper_dir = path
            break

print(f"📂 Checking scraper directory: {scraper_dir}")

if not os.path. exists(scraper_dir):
    print(f"❌ Directory not found:  {scraper_dir}")
    print(f"\n📂 Contents of {repo_dir}:")
    if os.path.exists(repo_dir):
        for item in os.listdir(repo_dir):
            print(f"   - {item}")
    raise FileNotFoundError("Scraper directory not found.  Check the repository download.")

# Change to scraper directory and add to Python path
os.chdir(scraper_dir)
if scraper_dir not in sys.path:
    sys.path. insert(0, scraper_dir)

print(f"✅ Working directory:  {os.getcwd()}")
print(f"✅ Python path updated")

# Now import local modules
import config
import utils

print(f"✅ Modules imported successfully")
print(f"   - Database: {config.db_name}")
print(f"   - Table: {config.table_name}")

# =============================================================================
# SCRAPER FUNCTIONS
# =============================================================================
async def extract_details_new(page):
    """Extract post details from Douyin page"""
    details = {
        "title": None,
        "content": None,
        "like_count": None,
        "comment_count": None,
        "share_count": None,
        "publish_time": None
    }

    try:
        title = await page.locator(
            'xpath=(//div[@data-e2e="user-info"]/div[2]/a/div)[2]'
        ).inner_text()
        details["title"] = title. split("\n")[0]
        print(f"✅ Title: {details['title']}")
    except Exception as e: 
        print(f"⚠️ Title error: {e}")

    try:
        details["content"] = await page.locator('xpath=//div[@data-e2e="detail-video-info"]/div[1]/div/h1').inner_text()
        print(f"✅ Content: {details['content'][: 50]}...")
    except Exception as e: 
        print(f"⚠️ Content error: {e}")

    try:
        details["like_count"] = await page.locator('xpath=//div[@data-e2e="detail-video-info"]/div[2]/div[1]/div[1]/span').inner_text()
        print(f"✅ Likes: {details['like_count']}")
    except Exception as e:
        print(f"⚠️ Like count error: {e}")

    try:
        details["comment_count"] = await page.locator('xpath=//div[@data-e2e="detail-video-info"]/div[2]/div/div[2]/span').inner_text()
        print(f"✅ Comments:  {details['comment_count']}")
    except Exception as e: 
        print(f"⚠️ Comment count error:  {e}")

    try:
        details["share_count"] = await page.locator('xpath=//div[@data-e2e="detail-video-info"]/div[2]/div/div[4]/span').inner_text()
        print(f"✅ Shares: {details['share_count']}")
    except Exception as e: 
        print(f"⚠️ Share count error:  {e}")

    try:
        publish_time = await page.locator('span[data-e2e="detail-video-publish-time"]').inner_text()
        publish_time = publish_time. replace('发布时间：', '').strip()
        dt_object = datetime. strptime(publish_time. strip(), '%Y-%m-%d %H:%M')
        details["publish_time"] = dt_object.strftime('%Y-%m-%d %H:%M:%S')
        print(f"✅ Publish time: {details['publish_time']}")
    except Exception as e: 
        print(f"⚠️ Publish time error:  {e}")

    return details


async def extract_comments(page, max_comments=500):
    """Extract comments from Douyin page"""
    print(f"📝 Extracting comments (max:  {max_comments})...")
    comments = []

    try: 
        for i in range(5):
            await page.evaluate("window. scrollBy(0, 800)")
            await page.wait_for_timeout(2000)
            if i % 2 == 0:
                print(f"   Scrolling...  {i+1}/5")

        comment_selectors = [
            '[data-e2e="comment-item"]',
            '. comment-item',
            '[class*="comment"]'
        ]

        comment_elements = None
        for selector in comment_selectors: 
            try: 
                comment_elements = await page.locator(selector).all()
                if comment_elements and len(comment_elements) > 0:
                    print(f"✅ Found {len(comment_elements)} comments")
                    break
            except: 
                continue

        if not comment_elements:
            print("⚠️ No comment elements found")
            return comments

        for idx, comment_elem in enumerate(comment_elements[: max_comments]):
            try:
                text = await comment_elem.inner_text()
                comments. append(text)
            except:
                continue

        print(f"✅ Extracted {len(comments)} raw comments")

    except Exception as e:
        print(f"❌ Comment extraction error:  {e}")

    return comments


async def scrape_douyin_post(url, conn):
    """Main scraping function using Playwright"""
    print("=" * 60)
    print("🚀 Launching Playwright browser...")
    print(f"🔗 Target URL: {url}")

    async with async_playwright() as p:
        browser = await p.chromium. launch(
            headless=False,
            args=['--start-maximized', '--disable-blink-features=AutomationControlled']
        )

        context = await browser.new_context(
            viewport={"width": 1920, "height":  1080},
            ignore_https_errors=True,
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )

        page = await context.new_page()

        try:
            await page.goto(url, wait_until="domcontentloaded")
            await page. wait_for_timeout(5000)

            try:
                close_btn = page.locator('xpath=//div[contains(text(), "登录后免费畅享高清视频")]/following-sibling::div[1]')
                if await close_btn. count() > 0:
                    await close_btn.click()
                    print("✅ Dismissed login popup")
                    await page.wait_for_timeout(2000)
            except:
                pass

            details = await extract_details_new(page)
            comments = await extract_comments(page, max_comments=20)
            parsed_comments = utils.extract_douyin_comments_from_text(comments)
            comments_json = json.dumps(parsed_comments, ensure_ascii=False) if parsed_comments else None

            item = [{
                'unnamed': None,
                'user_name': details['title']. strip() if details['title'] else None,
                'publication_date': details['publish_time']. strip() if details['publish_time'] else None,
                'content': details['content']. strip() if details['content'] else None,
                'shared_count': utils.chinese_unit_to_number(details['share_count']. strip()) if details['share_count'] else 0,
                'comment_count': utils. chinese_unit_to_number(details['comment_count'].strip()) if details['comment_count'] else 0,
                'like_count': utils.chinese_unit_to_number(details['like_count']. strip()) if details['like_count'] else 0,
                'link1': url,
                'link2': None,
                'content_segmented': None,
                'is_agriculture_related': None,
                'index_number': None,
                'comments':  comments_json
            }]

            utils.insert_data(conn, config.table_name, item)
            print("✅ Data saved to database!")

        except Exception as e: 
            print(f"❌ Error scraping:  {e}")

        finally:
            await browser.close()
            print("🔒 Browser closed")




📂 Checking scraper directory: C:\Users\wk2lam/Desktop\scrape_chinese_social_media\duoyin\scrape_chinese_social_media
✅ Working directory:  C:\Users\wk2lam\Desktop\scrape_chinese_social_media\duoyin\scrape_chinese_social_media
✅ Python path updated
✅ Modules imported successfully
   - Database: data.db
   - Table: posts


In [5]:
# =============================================================================
# MAIN EXECUTION
# =============================================================================
print("""
╔═════════════════════════════════════════════════════════════╗
║                   🎬 DOUYIN SCRAPER                         ║
╚═════════════════════════════════════════════════════════════╝
""")

# Connect to database
conn = sqlite3.connect(config.db_name)
print(f"✅ Connected to database: {config.db_name}")
utils.create_table(conn, config.table_name)

# Read URLs from urls.txt
urls_file = os.path.join(scraper_dir, 'urls.txt')
if os.path.exists(urls_file):
    with open(urls_file, 'r', encoding='utf-8') as f:
        urls = [line.strip() for line in f if line. strip()]
else:
    # Default test URL
    urls = ["https://www.douyin.com/video/7441961498318498084"]
    print("⚠️ No urls.txt found, using default URL")

print(f"📋 Found {len(urls)} URLs to scrape")

# Scrape each URL
for url in urls:
    if 'douyin. com/' in url or 'iesdouyin.com/' in url:
        try:
            asyncio.get_event_loop().run_until_complete(scrape_douyin_post(url, conn))
        except Exception as e: 
            print(f"❌ Error scraping {url}:  {e}")

conn.close()
print("\n" + "=" * 60)
print("✅ Douyin scraping complete!")


╔═════════════════════════════════════════════════════════════╗
║                   🎬 DOUYIN SCRAPER                         ║
╚═════════════════════════════════════════════════════════════╝

✅ Connected to database: data.db
📋 Found 2 URLs to scrape
🚀 Launching Playwright browser...
🔗 Target URL: https://www.iesdouyin.com/share/video/6862533859125251340/?region=CN&mid=6862534011185892109&u_code=0&did=MS4wLjABAAAA_oswJqZmU3zCGUyu8OVAC1UU2mCfmc4viCaQE_FNvdMR9u0Icg6C4KNfO5gmMhFF&iid=MS4wLjABAAAA0wvEGyff1ADjiPCblRQIWtvkG7eNV8LYyZ5_BPkYmSd4qCnN--z9LfPLty-TjygW&with_sec_did=1&titleType=title&share_sign=n02R2vZ5QCppS8kyhh0M5mtEAoEqelWcGWdKq65lS64-&share_version=110900&ts=1716365321&from_aid=2955&from_ssr=1
✅ Title: 桥城都匀
✅ Content: 一起来看看非洲刚果的工人们是如何分工的#农民工 #非洲 #效率...
✅ Likes: 45.0K
✅ Comments:  5385
✅ Shares: 3522
✅ Publish time: 2020-08-19 11:36:00
📝 Extracting comments (max:  20)...
   Scrolling...  1/5
   Scrolling...  3/5
   Scrolling...  5/5
✅ Found 115 comments
✅ Extracted 20 raw comment

In [6]:
# =============================================================================
# 📊 VIEW SCRAPED DATA
# =============================================================================
import sqlite3
import pandas as pd
import os

# Path to database
desktop_path = os. path.expanduser("~/Desktop")
db_path = os.path.join(desktop_path, "scrape_chinese_social_media", "duoyin", "scrape_chinese_social_media", "data.db")

# Connect and read data
conn = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM posts", conn)
conn.close()

print(f"📊 Total records scraped: {len(df)}")
print(f"\n📋 Columns: {list(df.columns)}")

# Display the data
df

📊 Total records scraped: 2

📋 Columns: ['id', 'unnamed', 'user_name', 'publication_date', 'content', 'shared_count', 'comment_count', 'like_count', 'link1', 'link2', 'content_segmented', 'is_agriculture_related', 'index_number', 'comments']


,id,unnamed,user_name,publication_date,content,shared_count,comment_count,like_count,link1,link2,content_segmented,is_agriculture_related,index_number,comments
0,1,None,桥城都匀,2020-08-19 11:36:00,一起来看看非洲刚果的工人们是如何分工的#农民工 #非洲 #效率,3522.0,5385.0,45000.0,https://www.iesdouyin.com/share/video/68625338...,None,None,None,None,"[{""username"": ""齐步走"", ""content"": ""故意的吧"", ""time""..."
1,2,None,宣威融媒,2022-07-27 17:23:00,云南又来上分了！农村喜事大家围着桌子一起打跳，透过屏幕满满的代入感。#结婚 #现场实拍 #云...,1180.0,3009.0,43300.0,https://www.iesdouyin.com/share/video/71249806...,None,None,None,None,"[{""username"": ""陈天睡大觉"", ""content"": ""作为汉族的我什么都不会..."


In [7]:
# =============================================================================
# 💾 EXPORT TO EXCEL
# =============================================================================
import pandas as pd
import os

desktop_path = os. path.expanduser("~/Desktop")
output_excel = os.path.join(desktop_path, "douyin_data.xlsx")

df.to_excel(output_excel, index=False)
print(f"✅ Data exported to:  {output_excel}")

✅ Data exported to:  C:\Users\wk2lam/Desktop\douyin_data.xlsx
